# 09 - CafeBERT standalone baseline

Train CafeBERT on the ensemble split (`load_splits()`), early-stop on val. Loops pipeline seeds 42, 131, 9998 (`TM_RUN_SEEDS` override) and writes `artifacts/seeds/{seed}/metrics/cafebert_standalone.json` (skip if that file exists).

Not part of the 5-model ensemble. After the loop, refreshes `artifacts/metrics/val_multiseed_summary.json` so CafeBERT validation means appear even if notebook 07 ran first.


In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy sentencepiece

In [ ]:
import sys, os

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

In [ ]:
from tm_research.ensemble.utils_io import load_splits
from tm_research.ensemble.bert_oof import BertOOFConfig, run_bert_fulltrain

train_df, val_df, test_df, label_map = load_splits()
print('train', train_df.shape, 'val', val_df.shape, 'test', test_df.shape)
print('classes', label_map.class_names)

In [ ]:
from tm_research.ensemble.utils_io import (
    activate_seed_artifacts,
    reset_artifacts_root,
    push_artifacts_to_persistent,
)
from tm_research.ensemble.multiseed import (
    resolve_pipeline_seeds,
    seed_is_complete,
    write_val_multiseed_summary,
)

SKIP_FILES = ['metrics/cafebert_standalone.json']
metrics = None
for seed in resolve_pipeline_seeds():
    if seed_is_complete(seed, SKIP_FILES):
        print(f'skip seed {seed}: {SKIP_FILES[0]} already present')
        continue
    activate_seed_artifacts(seed)
    try:
        print(f'Using seed: {seed}')
        cfg = BertOOFConfig(
            model_name='uitnlp/CafeBERT',
            output_name='cafebert_standalone',
            seed=seed,
            num_epochs=20,
            learning_rate=2e-5,
            max_length=256,
            train_batch_size=128,
            eval_batch_size=256,
            grad_accum_steps=2,
            warmup_ratio=0.1,
            weight_decay=0.01,
            early_stopping_patience=5,
            lr_scheduler_type='cosine',
            max_grad_norm=1.0,
            work_dir=str(paths.bert_work / f'cafebert_standalone_seed_{seed}'),
        )
        metrics = run_bert_fulltrain(cfg, train_df, val_df, test_df, label_map)
    finally:
        reset_artifacts_root()

summary_path = write_val_multiseed_summary()
print('wrote', summary_path)
push_artifacts_to_persistent()


In [ ]:
from sklearn.metrics import classification_report

if metrics is None:
    print('No new CafeBERT run this session (all seeds skipped). See val_multiseed_summary.json.')
else:
    print(
        f"acc={metrics['test_accuracy']:.4f}  "
        f"weighted_f1={metrics['test_f1_weighted']:.4f}  "
        f"macro_f1={metrics['test_f1']:.4f}"
    )
    print(classification_report(
        test_df['label'].map(label_map.label2id),
        metrics['test_pred'],
        target_names=label_map.class_names,
        zero_division=0,
    ))


In [ ]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()